## Notebook to show how to read a 3DSG from the 3DSSG dataset, print information and plot 3D


In [1]:
import os
import sys
import torch

os.environ["XDG_SESSION_TYPE"] = "x11"
os.environ["OPEN3D_DISABLE_WEB_VISUALIZER"] = "true"     

import numpy as np
import open3d as o3d

sys.path.append('../src')
from ssg import SceneGraph3D
from models import get_model_name, load_model, get_most_epochs_file
from data import QueryData

from encoders import get_node_encoder, get_edge_encoder, get_query_encoder

## Constants

In [2]:
from consts import DATA_DIR, OUTPUT_DIR

SCENE_DIR = DATA_DIR / "scene_graphs"
OUT_MODEL_DIR = OUTPUT_DIR / "models"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# Change this to change which scan you query
SCAN_UUID = "0cac7676-8d6f-2d13-8f3a-d7bf7f03e721"

EDGE_ENCODER = "all_minilm_l6v2"
NODE_ENCODER = "all_minilm_l6v2"
QUERY_ENCODER = "all_minilm_l6v2"
MULTI_ANSWER = True
MULTI_ANSWER_ARG = str(MULTI_ANSWER).lower()

# Change this to change which model you run
MODEL_SHORTHAND = "qmpn"

MODEL_NAME = get_model_name(
    MODEL_SHORTHAND,
    node_encoder=NODE_ENCODER,
    edge_encoder=EDGE_ENCODER,
    query_encoder=QUERY_ENCODER,
    multi=MULTI_ANSWER,
)

## Load the Scene Graph and Model

In [3]:
# Load scene graph
g = SceneGraph3D.from_json(SCENE_DIR / (SCAN_UUID + ".json"))
scan_id = g.scan_id
g.load_3d_data()

# Render scene graph to JSON and pdf
SceneGraph3D.render(g, "../outputs/example_plot")  # renders to example_plot.pdf
SceneGraph3D.to_json(g, "../outputs/example_output.json")


# Load model
model = load_model(MODEL_NAME)

most_trained = get_most_epochs_file(OUT_MODEL_DIR / MODEL_NAME)
model_state = torch.load(OUT_MODEL_DIR / MODEL_NAME / (str(most_trained) + ".pth"))
model.load_state_dict(model_state)

# Get encoders
encode_node = get_node_encoder(NODE_ENCODER)
encode_edge = get_edge_encoder(EDGE_ENCODER)
encode_query = get_query_encoder(QUERY_ENCODER)

# Embed scene
data, node_map, _ = SceneGraph3D.to_query_data(
    g, node_encoder=encode_node, edge_encoder=encode_edge, ret_node_maps=True
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Visualize 3D Object Semantic Segmented


In [6]:
vertex = g.pointcloud["vertex"]
xyz = np.vstack((vertex["x"], vertex["y"], vertex["z"])).T
colors = np.vstack((vertex["red"], vertex["green"], vertex["blue"])).T
obj_pcd = o3d.geometry.PointCloud()
obj_pcd.points = o3d.utility.Vector3dVector(xyz)
obj_pcd.colors = o3d.utility.Vector3dVector(colors / 255.0)
o3d.visualization.draw_plotly([obj_pcd])

### Visualize Mesh


In [17]:
o3d.visualization.draw([{"geometry": g.mesh, "name": "3RScan Mesh"}], show_ui=True)

[Open3D INFO] [o3d] name: Room Scene.png.


### Model Helper Functions

In [58]:
@torch.no_grad()
def run_model(model, node_map, query_data):
    model.eval()
    out = model(
        x=query_data.x,
        edge_index=query_data.edge_index,
        edge_attr=query_data.edge_attr,
        query=query_data.query.unsqueeze(0),
        batch=torch.zeros(query_data.num_nodes, dtype=torch.int32),
    )
    return [(node_map[i], out[i].item()) for i in torch.argsort(out, descending=True)]

def get_selected(answer, multi=True, threshold=0.5):
    if multi:
        return list(filter(lambda out: out[1] >=threshold, answer))

def visualise_selected(ids):
    meshes = g.get_colored_objects([int(id[0]) for id in ids], [[(1 + id[1]) / 2, 0.0, 0.0] for id in ids])
    selected_meshes = [{"geometry": meshes[i + 1], "name": f"Object {ids[i]}"} for i in range(len(ids))]
    o3d.visualization.draw([{"geometry": g.mesh, "name": "3RScan Mesh"}, *selected_meshes], show_ui=True)

VISUALISE = False

def query_model(model, node_map, query, limit=5, threshold=0.5, visualise=VISUALISE, multi=True):
    query_data = QueryData(
        x=data.x,
        pos=data.pos,
        edge_index=data.edge_index,
        edge_attr=data.edge_attr,
        query=encode_query([query])[0],
    )

    output = run_model(model, node_map, query_data)

    if visualise:
        selected = get_selected(output, multi=multi, threshold=threshold)
        visualise_selected(selected)

    if limit != None:
        output = output[:limit]
    return output

## Non-Relational Questions

In [18]:
# 1
query_model(model, node_map, "Which object is a bed?")  # Expecting 2

[('2', 0.29085445404052734),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('42', 0.08910531550645828)]

In [19]:
# 2
query_model(model, node_map, "Which object is a floor?")  # Expecting 1

[('40', 0.9926090836524963),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('17', 0.08675113320350647)]

In [20]:
# 3
query_model(model, node_map, "Where can I sit?")  # Expecting 2, 6, 56

[('2', 0.9926090836524963),
 ('56', 0.9926090836524963),
 ('6', 0.9926090836524963),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495)]

In [21]:
# 4
query_model(model, node_map, "What can light up the room?")  # Expecting 51, 52, 53

[('52', 0.9926090836524963),
 ('19', 0.9639877676963806),
 ('51', 0.8965842127799988),
 ('35', 0.5065181255340576),
 ('53', 0.4769138693809509)]

In [38]:
# 5
query_model(model, node_map, "What is tall and narrow?", limit=15)  # Expecting 28

[('43', 0.9926090836524963),
 ('4', 0.9926090836524963),
 ('39', 0.9926090836524963),
 ('32', 0.9926090836524963),
 ('28', 0.9926090836524963),
 ('27', 0.9926090836524963),
 ('25', 0.9926090836524963),
 ('22', 0.9926090836524963),
 ('20', 0.9891856908798218),
 ('51', 0.9868428111076355),
 ('100', 0.9078575968742371),
 ('56', 0.9019261002540588),
 ('24', 0.6675586104393005),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495)]

In [23]:
# 6
query_model(model, node_map, "Which object is a shelf?") # Expecting 55

[('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('1', 0.10210643708705902),
 ('39', 0.09858091175556183)]

In [24]:
# 7
query_model(model, node_map, "What could I warm myself with?")  # Expecting 3

[('3', 0.9926090836524963),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('2', 0.08053622394800186)]

In [25]:
# 8
query_model(model, node_map, "What is a cabinet?")  # Expecting 29, 37

[('4', 0.9926090836524963),
 ('29', 0.9926090836524963),
 ('37', 0.942245602607727),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495)]

In [26]:
# 9
query_model(model, node_map, "Where could I put some clothes?", limit=10)  # Expecting 4, 28, 29, 37

[('4', 0.9926090836524963),
 ('28', 0.9514626860618591),
 ('29', 0.8749293088912964),
 ('38', 0.32653695344924927),
 ('41', 0.2545417845249176),
 ('10', 0.23476232588291168),
 ('2', 0.2012859731912613),
 ('8', 0.17529930174350739),
 ('39', 0.15118856728076935),
 ('55', 0.11797914654016495)]

In [40]:
# 10
query_model(model, node_map, "Where could I put a picture?", limit=15)  # Expecting 5, 7, 10, 13, 14, 27, 28, 29, 37, 55

[('13', 0.9926090836524963),
 ('24', 0.9926090836524963),
 ('41', 0.9926090836524963),
 ('25', 0.9926090836524963),
 ('43', 0.9926090836524963),
 ('14', 0.9926090836524963),
 ('7', 0.9926090836524963),
 ('38', 0.9926090836524963),
 ('5', 0.9921395182609558),
 ('10', 0.9900804162025452),
 ('44', 0.44317013025283813),
 ('29', 0.2257416844367981),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('39', 0.1168961226940155)]

## Relational Questions

In [43]:
# 1
query_model(model, node_map, "What is left of the bed?")  # Expectin 28, 51

[('2', 0.9926090836524963),
 ('3', 0.9579615592956543),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703)]

In [46]:
# 2
query_model(model, node_map, "What is the shelf attached to?")  # Expecting 37

[('24', 0.9926090836524963),
 ('4', 0.9740516543388367),
 ('29', 0.8270987868309021),
 ('8', 0.17529930174350739),
 ('25', 0.17339317500591278)]

In [47]:
# 3
query_model(model, node_map, "What is a cabinet attached too?")  # Expecting 10

[('29', 0.9926090836524963),
 ('4', 0.9926090836524963),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('39', 0.10760603100061417)]

In [48]:
# 4
query_model(model, node_map, "What is close to the bed?")  # Expecting 28

[('2', 0.9926090836524963),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('39', 0.10843592137098312),
 ('18', 0.10381976515054703)]

In [49]:
# 5
query_model(model, node_map, "What is on the bed?", limit=10)  # Expecting 3, 15, 20, 21, 22, 100

[('3', 0.9926090836524963),
 ('100', 0.9926090836524963),
 ('21', 0.9926090836524963),
 ('15', 0.9926090836524963),
 ('20', 0.9634259939193726),
 ('22', 0.9633034467697144),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('1', 0.1041126549243927),
 ('18', 0.10381976515054703)]

In [50]:
# 6
query_model(model, node_map, "What's on the table?", limit=15)  # Expecting 24, 25, 26

[('26', 0.9926090836524963),
 ('44', 0.9926090836524963),
 ('43', 0.9926090836524963),
 ('42', 0.9926090836524963),
 ('41', 0.9926090836524963),
 ('30', 0.9926090836524963),
 ('38', 0.9926090836524963),
 ('33', 0.9926090836524963),
 ('32', 0.9926090836524963),
 ('31', 0.9926090836524963),
 ('21', 0.7314167022705078),
 ('100', 0.6701465845108032),
 ('20', 0.5609846115112305),
 ('15', 0.4391064941883087),
 ('8', 0.17529930174350739)]

In [54]:
# 7
query_model(model, node_map, "What's behind the nightstand?", limit=10)  # Expecting 37, 39, 56

[('53', 0.7300530672073364),
 ('2', 0.2969334125518799),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('39', 0.09606355428695679),
 ('27', 0.09033714979887009),
 ('51', 0.08721589297056198),
 ('40', 0.08559150248765945),
 ('37', 0.08414780348539352)]

In [ ]:
# 8
query_model(model, node_map, "What's behind the table?", limit=10)  # Expecting 40

[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


[('40', 0.9926090836524963),
 ('53', 0.9926090836524963),
 ('39', 0.971261739730835),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('29', 0.083771251142025),
 ('5', 0.07492582499980927),
 ('10', 0.07462810724973679),
 ('51', 0.0741986408829689)]

In [60]:
# 9
query_model(model, node_map, "What's attached to the window?", limit=10)  # Expecting 19, 35

[('35', 0.9926090836524963),
 ('19', 0.9926090836524963),
 ('24', 0.9494685530662537),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('38', 0.0835680365562439),
 ('41', 0.08351989090442657),
 ('44', 0.08207899332046509),
 ('32', 0.08044430613517761)]

In [ ]:
# 10
query_model(model, node_map, "What's between the bed and a cabinet?", limit=10)  # Expecting 6, 56

[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDuplicatedTriangles] This mesh contains triangle uvs that are not handled in this function
[Open3D WARNING] [RemoveDegenerateTriangles] This mesh contains triangle uvs that are not handled in this function


[('2', 0.9926090836524963),
 ('56', 0.951892077922821),
 ('39', 0.8763079643249512),
 ('8', 0.17529930174350739),
 ('6', 0.1236272007226944),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('28', 0.09654872864484787),
 ('3', 0.09623590111732483),
 ('31', 0.08716778457164764)]

## Complex

In [ ]:
query_model(model, node_map, "I want to do some paper work. Where could I do it?")  # Expecting 27

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[('2', 0.9808470606803894),
 ('28', 0.9579182863235474),
 ('27', 0.9351805448532104),
 ('37', 0.650383710861206),
 ('56', 0.5445812344551086)]

In [ ]:
query_model(model, node_map, "What can I close to block the sunlight?") # Expecting 19, 35

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[('19', 0.9778734445571899),
 ('35', 0.6869725584983826),
 ('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703)]

In [ ]:
query_model(model, node_map, "Where could I put my drink?", limit=10)  # Expecting 27

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[('4', 0.9926090836524963),
 ('29', 0.9926090836524963),
 ('28', 0.9926090836524963),
 ('14', 0.6440980434417725),
 ('13', 0.633219301700592),
 ('6', 0.6244866251945496),
 ('27', 0.272981196641922)]

## Non-Questions

In [127]:
query_model(model, node_map, "right")[:5]  # Expecting nothing

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[('8', 0.17529930174350739),
 ('55', 0.11797914654016495),
 ('18', 0.10381976515054703),
 ('2', 0.09376417100429535),
 ('42', 0.0911199077963829)]